# **Funnel Analysis: Product Performance Analysis & Conversion Rate Optimization**

Objective: Using data to identify which products are performing best and to find bottlenecks in the purchasing process for each product category.

## **I. Set Up & Data Ingestion**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")
#
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
order_items = pd.read_csv('/content/drive/MyDrive/Analytics Report/Data/order_items.csv')
order_item_refunds = pd.read_csv('/content/drive/MyDrive/Analytics Report/Data/order_item_refunds.csv')
products = pd.read_csv('/content/drive/MyDrive/Analytics Report/Data/products.csv')
website_pageview = pd.read_csv('/content/drive/MyDrive/Analytics Report/Data/website_pageviews.csv')
website_sessions = pd.read_csv('/content/drive/MyDrive/Analytics Report/Data/website_sessions.csv')

## **II. Revenue & Refund Analysis**

In [ ]:
# Merging dataframes
orders = pd.merge(order_items, products, on = 'product_id', how = 'left')
order_summary = pd.merge(orders, order_item_refunds[['order_item_id', 'refund_amount_usd']], on = 'order_item_id', how = 'left')

# Handling null
order_summary['refund_amount_usd'] = order_summary['refund_amount_usd'].fillna(0)
order_summary['is_refunded'] = order_summary['refund_amount_usd'] > 0
order_summary['net_revenue'] = order_summary['price_usd'] - order_summary['refund_amount_usd']

# Group by products
product_perf = order_summary.groupby('product_name').agg(
    total_revenue = ('price_usd', 'sum'),
    total_refund = ('refund_amount_usd', 'sum'),
    net_revenue = ('net_revenue', 'sum'),
    total_orders = ('order_item_id', 'count'),
    refund_orders = ('is_refunded', 'sum')
).reset_index()

# Refund rate
product_perf['refund_rate'] = (product_perf['refund_orders'] / product_perf['total_orders']) * 100

# Rank by net revenue
product_perf = product_perf.sort_values(by = 'net_revenue', ascending = False)

product_perf


In [ ]:
# Visualize Net Revenue Comparison
plt.figure(figsize=(10, 5))
sns.barplot(x='net_revenue', y='product_name', data=product_perf, palette='viridis')
plt.title('Net Revenue Comparison')
plt.xlabel('million USD')
plt.ylabel('')
plt.show()

# Visualize Refund Rate by products
plt.figure(figsize=(10, 5))
sns.barplot(x='refund_rate', y='product_name', data=product_perf, palette='Reds_r')
plt.title('Refund Rate by products')
plt.xlabel('Percent (%)')
plt.ylabel('')
plt.show()

### Evaluate net revenue and refund rate

- The Original Mr. Fuzzy is currently generating the highest net revenue.
- The Birthday Sugar Panda has the highest return rate. This requires a review of the manufacturing process or the product description on the website to ensure it accurately reflects the physical item.

## **III. Product-Specific Funnel**

In [ ]:
# Inner Join website_sessions and website_pageviews to filter arbitrarily by utm_source và device_type
web_check = pd.merge(website_sessions, website_pageview, on = 'website_session_id', how = 'inner')
web_check['utm_source'].fillna('organic', inplace = True) # handling null in utm_source

web_check.duplicated().sum()

In [ ]:
# URLs
product_pages = {
    'Mr. Fuzzy': '/the-original-mr-fuzzy',
    'Love Bear': '/the-forever-love-bear',
    'Sugar Panda': '/the-birthday-sugar-panda',
    'Mini Bear': '/the-hudson-river-mini-bear'
}

def analyze_product_funnel(df, product_name, product_url, utm_source=None, device_type=None):
    """
    This function analyzes the conversion funnel of a product and returns a summary DataFrame.
    Supporting arbitrary filtering by utm_source and device type.
    """
    # 1. Make a copy
    filtered_df = df.copy()

    # 2. Apply filters if parameters are passed in
    if utm_source:
        filtered_df = filtered_df[filtered_df['utm_source'] == utm_source]

    if device_type:
        filtered_df = filtered_df[filtered_df['device_type'] == device_type]

    # 3. Calculate the number of sessions at each step of the funnel based on the filtered data

    # Step 1: Sessions landed in /products
    sessions_at_products = filtered_df[filtered_df['pageview_url'] == '/products']['website_session_id'].unique()
    relevant_df = filtered_df[filtered_df['website_session_id'].isin(sessions_at_products)]

    # Step 2: Sessions landed in product details
    sessions_at_detail = relevant_df[relevant_df['pageview_url'] == product_url]['website_session_id'].unique()

    # Step 3: Sessions landed in add-to-cart
    sessions_at_cart = relevant_df[(relevant_df['website_session_id'].isin(sessions_at_detail)) &
                                  (relevant_df['pageview_url'] == '/cart')]['website_session_id'].unique()

    # Step 4: Sessions purchased successfully (/thank-you-for-your-order)
    sessions_at_thanks = relevant_df[(relevant_df['website_session_id'].isin(sessions_at_cart)) &
                                    (relevant_df['pageview_url'] == '/thank-you-for-your-order')]['website_session_id'].unique()

    # 4. Summary
    summary = pd.DataFrame({
        'Step': ['1. Browse Products', '2. View Detail', '3. Add to Cart', '4. Purchase'],
        'Sessions': [len(sessions_at_products), len(sessions_at_detail),
                     len(sessions_at_cart), len(sessions_at_thanks)]
    })

    # 5. CTR between steps
    summary['CTR'] = (summary['Sessions'] / summary['Sessions'].shift(1)).fillna(1.0)
    return summary

In [ ]:

# "The Original Mr. Fuzzy" & "The Forever Love Bear" funnel
fuzzy_funnel_all = analyze_product_funnel(web_check, 'Mr. Fuzzy', product_pages['Mr. Fuzzy'])
bear_funnel_all = analyze_product_funnel(web_check, 'Love Bear', product_pages['Love Bear'])

print("The Original Mr. Fuzzy funnel:")
display(fuzzy_funnel_all)
print("\nThe Forever Love Bear funnel:")
display(bear_funnel_all)

In [ ]:
# Visualize CTR Comparison
plt.figure(figsize=(12, 6))
plt.plot(fuzzy_funnel_all['Step'], fuzzy_funnel_all['CTR'], marker='o', label='Mr. Fuzzy', linewidth=2)
plt.plot(bear_funnel_all['Step'], bear_funnel_all['CTR'], marker='s', label='Love Bear', linewidth=2)
plt.title('Step-to-Step CTR Comparison')
plt.ylabel('CTR compared to previous step')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### Evaluating Product-Specific Funnel

- Customers show an overwhelmingly higher interest in Mr. Fuzzy. The CTR to Mr. Fuzzy's product detail page reached over 60%, whereas Love Bear only stood at around 10%.

- The conversion rate for both products is comparable (both hovering above 30%).

It is necessary to review how Love Bear is displayed on the product category page. The add-to-cart rate for this product, once customers view its details, is actually higher than that of Mr. Fuzzy. This suggests that the current thumbnail might lack visual appeal at a glance, or the product is placed in an inconspicuous position on the page.

Next, the checkout process needs to be re-evaluated. Since the drop-off rate (cart abandonment) for both products is remarkably similar, there is a strong suspicion of a systemic, platform-wide issue occurring at the payment stage.

In [ ]:
# Conversion funnel for products by device
fuzzy_mobile = analyze_product_funnel(web_check, 'Mr. Fuzzy', product_pages['Mr. Fuzzy'], device_type= 'mobile')
fuzzy_desktop = analyze_product_funnel(web_check, 'Mr. Fuzzy', product_pages['Mr. Fuzzy'], device_type= 'desktop')
love_bear_mobile = analyze_product_funnel(web_check, 'Love Bear', product_pages['Love Bear'], device_type= 'mobile')
love_bear_desktop = analyze_product_funnel(web_check, 'Love Bear', product_pages['Love Bear'], device_type= 'desktop')

In [ ]:
# Comparing conversion funnel between fuzzy_mobile and fuzzy_desktop

# Define stages (steps) and extract session counts for plotly funnel
stages = fuzzy_mobile['Step'].tolist()
sessions_mobile = fuzzy_mobile['Sessions'].tolist()
sessions_desktop = fuzzy_desktop['Sessions'].tolist()

# Create a figure with two subplots side-by-side with funnel specs
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Mr. Fuzzy Funnel (Desktop)", "Mr. Fuzzy Funnel (Mobile)"),
    specs=[[{"type": "funnel"}, {"type": "funnel"}]]
)

# Add Funnel chart for Desktop (Column 1)
fig.add_trace(go.Funnel(
    name='Desktop',
    y=stages,
    x=sessions_desktop,
    textinfo="value+percent initial+percent previous",
    marker={"color": "#6CB4EE"} # Light blue color
), row=1, col=1)

# Add Funnel chart for Mobile (Column 2)
fig.add_trace(go.Funnel(
    name='Mobile',
    x=sessions_mobile,
    textinfo="value+percent initial+percent previous",
    marker={"color": "#F4B28F"} # Light orange color
), row=1, col=2)


# Configure overall title and layout
fig.update_layout(
    title_text='<b>Conversion Funnel Comparison</b><br><sup>Visualize CR on each device when purchasing Mr. Fuzzy.</sup>',
    height=600,
    showlegend=False,
    font_size=12
)

fig.update_yaxes(showticklabels=False, row=1, col=2)

fig.show()

In [ ]:
# Comparing conversion funnel between love_bear_mobile and love_bear_desktop

# Define stages (steps) and extract session counts for plotly funnel
stages = love_bear_mobile['Step'].tolist()
sessions_mobile = love_bear_mobile['Sessions'].tolist()
sessions_desktop = love_bear_desktop['Sessions'].tolist()

# Create a figure with two subplots side-by-side with funnel specs
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("The Forever Love Bear (Desktop)", "The Forever Love Bear (Mobile)"),
    specs=[[{"type": "funnel"}, {"type": "funnel"}]]
)

# Add Funnel chart for Desktop (Column 1)
fig.add_trace(go.Funnel(
    name='Desktop',
    y=stages,
    x=sessions_desktop,
    textinfo="value+percent initial+percent previous",
    marker={"color": "#6CB4EE"} # Light blue color
), row=1, col=1)

# Add Funnel chart for Mobile (Column 2)
fig.add_trace(go.Funnel(
    name='Mobile',
    x=sessions_mobile,
    textinfo="value+percent initial+percent previous",
    marker={"color": "#F4B28F"} # Light orange color
), row=1, col=2)


# Configure overall title and layout
fig.update_layout(
    title_text='<b>Conversion Funnel Comparison</b><br><sup>Visualize CR on each device when purchasing The Forever Love Bear.</sup>',
    height=600,
    showlegend=False,
    font_size=12
)

fig.update_yaxes(showticklabels=False, row=1, col=2)

fig.show()

### Evaluating the conversion funnel for products by device

The checkout issue appears to be more severe on mobile devices.

The CTR to view Mr. Fuzzy's product details reached 66% on Desktop, whereas it only hit 51% on Mobile. This 15% gap suggests that the product listing interface on mobile screens might be diminishing visual attention or creating usability friction for customers.

The exceptionally low CTR (10%) for Love Bear occurs consistently across both Desktop and Mobile. This firmly corroborates the previous assessment: the thumbnail image, product naming, or positioning of Love Bear on the category page lacks visual appeal, and this is a device-agnostic issue.

## **IV. Business Recommendation**

- **Priority 1:** Immediately Fix the Mobile Checkout Flow. The UI/UX and engineering teams need to conduct end-to-end user testing by role-playing as customers navigating the mobile checkout process. Investigate common friction points such as excessively long input forms, obscured "Checkout" buttons, slow page load speeds, or integration errors with e-wallets/banking apps.

- **Priority 2:** Revamp Love Bear's Visual Merchandising. Since Love Bear's View Detail to Add-to-Cart conversion rate is excellent (Desktop 58%, Mobile 51%), the bottleneck lies entirely in the lack of initial clicks. Implement A/B testing to evaluate new thumbnail images and modify other UI display elements related to this product's visibility.

- **Priority 3:** Optimize the Mobile Product Listing Page. Re-evaluate image sizes, button spacing (touch targets), and the overall layout of the product cards on the mobile interface. The goal is to ensure the mobile browsing and clicking experience is as seamless as on desktop, aiming to pull Mr. Fuzzy's View Detail rate back above the 60% threshold.